In [1]:
import os
import time
import wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import wavfile
from scipy.fft import rfft
from scipy.signal import get_window
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder


In [2]:
DATA_DIR = r"C:\Users\bhavy\OneDrive\Documents\ASV_2017\baseline_CM\baseline_CM\ASVspoof2017_V2_train"
PROTOCOL_DIR = r"C:\Users\bhavy\OneDrive\Documents\ASV_2017\baseline_CM\baseline_CM\protocol_V2"

AUDIO_EXTENSIONS = {".wav", ".flac"}
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_TIMING_RUNS = 10
K_VALUES = list(range(1, 16, 2))
BASE_K = 3


In [3]:
def find_audio_files(data_dir):
    files = []
    for root, _, names in os.walk(data_dir):
        for name in names:
            if os.path.splitext(name)[1].lower() in AUDIO_EXTENSIONS:
                files.append(os.path.join(root, name))
    return sorted(files)

def find_protocol_files(protocol_dir):
    files = []
    for root, _, names in os.walk(protocol_dir):
        for name in names:
            files.append(os.path.join(root, name))
    return sorted(files)

audio_files = find_audio_files(DATA_DIR)
protocol_files = find_protocol_files(PROTOCOL_DIR)

print(len(audio_files))
print(protocol_files)


3014
['C:\\Users\\bhavy\\OneDrive\\Documents\\ASV_2017\\baseline_CM\\baseline_CM\\protocol_V2\\ASVspoof2017_V2_dev.trl.txt', 'C:\\Users\\bhavy\\OneDrive\\Documents\\ASV_2017\\baseline_CM\\baseline_CM\\protocol_V2\\ASVspoof2017_V2_eval.trl.txt', 'C:\\Users\\bhavy\\OneDrive\\Documents\\ASV_2017\\baseline_CM\\baseline_CM\\protocol_V2\\ASVspoof2017_V2_train.trn.txt']


In [4]:
def read_protocol(protocol_files):
    rows = []
    for path in protocol_files:
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    if len(parts) >= 2:
                        rows.append(parts)
        except Exception:
            pass
    return rows

protocol_rows = read_protocol(protocol_files)
len(protocol_rows)


18030

In [5]:
def build_label_map(protocol_rows):
    label_map = {}
    for parts in protocol_rows:
        tokens = [x.lower() for x in parts]
        label = None
        if "genuine" in tokens:
            label = 0
        elif "spoof" in tokens:
            label = 1
        elif "bonafide" in tokens:
            label = 0
        elif "fake" in tokens:
            label = 1
        if label is None:
            continue
        candidates = [x for x in parts if x.lower().endswith((".wav", ".flac"))]
        if not candidates:
            candidates = [x for x in parts if not x.replace(".", "", 1).isdigit()]
        if candidates:
            key = os.path.splitext(os.path.basename(candidates[0]))[0].lower()
            label_map[key] = label
    return label_map

label_map = build_label_map(protocol_rows)
len(label_map)


18030

In [6]:
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def mfcc_features(path, n_mfcc=13, n_fft=512, hop_length=256, n_mels=26):
    sr, y = wavfile.read(path)
    y = np.asarray(y, dtype=np.float64)

    if y.ndim > 1:
        y = np.mean(y, axis=1)

    if y.size == 0:
        return np.zeros(n_mfcc * 2, dtype=np.float64)

    y = y / (np.max(np.abs(y)) + 1e-12)

    frame_length = min(n_fft, max(32, y.size))
    hop = min(hop_length, frame_length)
    if y.size < frame_length:
        y = np.pad(y, (0, frame_length - y.size))

    n_frames = 1 + max(0, (y.size - frame_length) // hop)
    if n_frames < 1:
        n_frames = 1

    frames = np.zeros((n_frames, frame_length), dtype=np.float64)
    window = get_window("hamming", frame_length, fftbins=True)

    for i in range(n_frames):
        start = i * hop
        segment = y[start:start + frame_length]
        if segment.size < frame_length:
            segment = np.pad(segment, (0, frame_length - segment.size))
        frames[i] = segment * window

    spec = np.abs(rfft(frames, n=frame_length, axis=1)) ** 2

    low_mel = hz_to_mel(0)
    high_mel = hz_to_mel(sr / 2.0)
    mel_points = np.linspace(low_mel, high_mel, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((frame_length + 1) * hz_points / sr).astype(int)
    bins = np.clip(bins, 0, spec.shape[1] - 1)

    fb = np.zeros((n_mels, spec.shape[1]))
    for m in range(1, n_mels + 1):
        left, center, right = bins[m - 1], bins[m], bins[m + 1]
        if center <= left:
            center = min(left + 1, spec.shape[1] - 1)
        if right <= center:
            right = min(center + 1, spec.shape[1] - 1)

        if center > left:
            fb[m - 1, left:center] = (np.arange(left, center) - left) / (center - left)
        if right > center:
            fb[m - 1, center:right] = (right - np.arange(center, right)) / (right - center)

    mel_energy = np.maximum(spec @ fb.T, 1e-12)
    log_mel = np.log(mel_energy)

    n = np.arange(n_mels)
    k = np.arange(n_mfcc)[:, None]
    dct = np.cos(np.pi / n_mels * (n + 0.5) * k)
    coeffs = log_mel @ dct.T

    return np.concatenate([coeffs.mean(axis=0), coeffs.std(axis=0)])

def audio_key(path):
    return os.path.splitext(os.path.basename(path))[0].lower()


In [7]:
def infer_label_from_protocol(path, label_map):
    key = audio_key(path)
    if key in label_map:
        return label_map[key]
    return None

X_list = []
y_list = []
used_files = []

for i, path in enumerate(audio_files):
    label = infer_label_from_protocol(path, label_map)
    if label is None:
        continue
    try:
        feat = mfcc_features(path)
        if np.all(np.isfinite(feat)):
            X_list.append(feat)
            y_list.append(label)
            used_files.append(path)
    except Exception:
        continue

X = np.asarray(X_list, dtype=np.float64)
y = np.asarray(y_list, dtype=np.int64)

print(X.shape)
print(y.shape)
print(np.unique(y, return_counts=True))


(3014, 26)
(3014,)
(array([0, 1], dtype=int64), array([1507, 1507], dtype=int64))


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

train_mean = np.nanmean(X_train, axis=0)
train_mean = np.where(np.isfinite(train_mean), train_mean, 0.0)

X_train = np.where(np.isfinite(X_train), X_train, train_mean)
X_test = np.where(np.isfinite(X_test), X_test, train_mean)

train_std = np.std(X_train, axis=0)
train_std[train_std == 0] = 1.0

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print(X_train_scaled.shape)
print(X_test_scaled.shape)


(2411, 26)
(603, 26)


In [9]:
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def bubble_sort(values):
    values = list(values)
    n = len(values)
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if values[j][0] > values[j + 1][0]:
                values[j], values[j + 1] = values[j + 1], values[j]
                swapped = True
        if not swapped:
            break
    return values

def selection_sort(values):
    values = list(values)
    n = len(values)
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            if values[j][0] < values[min_idx][0]:
                min_idx = j
        values[i], values[min_idx] = values[min_idx], values[i]
    return values

def insertion_sort(values):
    values = list(values)
    for i in range(1, len(values)):
        current = values[i]
        j = i - 1
        while j >= 0 and values[j][0] > current[0]:
            values[j + 1] = values[j]
            j -= 1
        values[j + 1] = current
    return values

SORTERS = {
    "bubble": bubble_sort,
    "selection": selection_sort,
    "insertion": insertion_sort
}

class MyKNN:
    def __init__(self, k=3, sorting="insertion"):
        self.k = k
        self.sorting = sorting
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def _predict_one(self, x):
        distances = [(euclidean_distance(x, xi), yi) for xi, yi in zip(self.X_train, self.y_train)]
        nearest = SORTERS[self.sorting](distances)[:self.k]
        counts = {}
        for _, label in nearest:
            counts[label] = counts.get(label, 0) + 1
        return sorted(counts.items(), key=lambda z: (-z[1], z[0]))[0][0]

    def predict(self, X):
        return np.asarray([self._predict_one(x) for x in np.asarray(X, dtype=float)])

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


In [10]:
class AIKNN:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            idx = np.argsort(distances, kind="stable")[:self.k]
            labels = self.y_train[idx]
            values, counts = np.unique(labels, return_counts=True)
            predictions.append(values[np.argmax(counts)])
        return np.asarray(predictions)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

class WeightedKNN:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            idx = np.argsort(distances, kind="stable")[:self.k]
            labels = self.y_train[idx]
            d = distances[idx]
            weights = 1.0 / (d + 1e-12)
            scores = {}
            for label, weight in zip(labels, weights):
                scores[label] = scores.get(label, 0.0) + weight
            predictions.append(sorted(scores.items(), key=lambda z: (-z[1], z[0]))[0][0])
        return np.asarray(predictions)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


In [11]:
def metric_summary(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

manual = MyKNN(k=BASE_K, sorting="insertion").fit(X_train_scaled, y_train)
ai_model = AIKNN(k=BASE_K).fit(X_train_scaled, y_train)
sk_model = KNeighborsClassifier(n_neighbors=BASE_K, weights="uniform").fit(X_train_scaled, y_train)

start = time.perf_counter()
manual_pred = manual.predict(X_test_scaled)
manual_time = time.perf_counter() - start

start = time.perf_counter()
sk_pred = sk_model.predict(X_test_scaled)
sk_time = time.perf_counter() - start

start = time.perf_counter()
ai_pred = ai_model.predict(X_test_scaled)
ai_time = time.perf_counter() - start

results = pd.DataFrame([
    {"Model": "User-written kNN", **metric_summary(y_test, manual_pred), "Time (s)": manual_time},
    {"Model": "Scikit-Learn kNN", **metric_summary(y_test, sk_pred), "Time (s)": sk_time},
    {"Model": "GenAI kNN", **metric_summary(y_test, ai_pred), "Time (s)": ai_time}
])

results


,Model,Accuracy,Precision,Recall,F1-Score,Time (s)
0,User-written kNN,0.996683,0.996678,0.996678,0.996678,310.645773
1,Scikit-Learn kNN,0.996683,0.996678,0.996678,0.996678,0.610244
2,GenAI kNN,0.996683,0.996678,0.996678,0.996678,0.542634


In [ ]:
def timed_evaluation(model_factory, runs=N_TIMING_RUNS):
    times = []
    for _ in range(runs):
        model = model_factory()
        start = time.perf_counter()
        model.fit(X_train_scaled, y_train)
        model.predict(X_test_scaled)
        times.append(time.perf_counter() - start)
    return float(np.mean(times))

timing_results = pd.DataFrame([
    ["User-written kNN", timed_evaluation(lambda: MyKNN(k=BASE_K, sorting="insertion"))],
    ["Scikit-Learn kNN", timed_evaluation(lambda: KNeighborsClassifier(n_neighbors=BASE_K))],
    ["GenAI kNN", timed_evaluation(lambda: AIKNN(k=BASE_K))]
], columns=["Model", "Average Time over 10 Runs (s)"])

results = results.drop(columns=["Time (s)"]).merge(timing_results, on="Model")
results


In [ ]:
k_results = []

for k in K_VALUES:
    for name, factory in [
        ("User-written kNN", lambda k=k: MyKNN(k=k, sorting="insertion")),
        ("Scikit-Learn kNN", lambda k=k: KNeighborsClassifier(n_neighbors=k)),
        ("GenAI kNN", lambda k=k: AIKNN(k=k))
    ]:
        model = factory()
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
        k_results.append({
            "k": k,
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred)
        })

k_results = pd.DataFrame(k_results)
k_results


In [ ]:
plt.figure(figsize=(8, 5))
for name in k_results["Model"].unique():
    subset = k_results[k_results["Model"] == name]
    plt.plot(subset["k"], subset["Accuracy"], marker="o", label=name)
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.title("kNN Accuracy for Different k Values")
plt.xticks(K_VALUES)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
weighted_results = []

for k in K_VALUES:
    model = WeightedKNN(k=k).fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    weighted_results.append({
        "k": k,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0)
    })

weighted_results = pd.DataFrame(weighted_results)
weighted_results


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(weighted_results["k"], weighted_results["Accuracy"], marker="o", label="Weighted kNN")
plt.plot(
    k_results[k_results["Model"] == "GenAI kNN"]["k"],
    k_results[k_results["Model"] == "GenAI kNN"]["Accuracy"],
    marker="s",
    label="Unweighted GenAI kNN"
)
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.title("Weighted and Unweighted kNN Accuracy")
plt.xticks(K_VALUES)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import unittest

class TestKNN(unittest.TestCase):
    def setUp(self):
        self.X = np.array([[0., 0.], [0., 1.], [5., 5.], [5., 6.]])
        self.y = np.array([0, 0, 1, 1])

    def test_distance(self):
        self.assertAlmostEqual(euclidean_distance(np.array([0., 0.]), np.array([3., 4.])), 5.0)

    def test_bubble_sort(self):
        self.assertEqual([x[0] for x in bubble_sort([(3, 0), (1, 0), (2, 0)])], [1, 2, 3])

    def test_selection_sort(self):
        self.assertEqual([x[0] for x in selection_sort([(3, 0), (1, 0), (2, 0)])], [1, 2, 3])

    def test_insertion_sort(self):
        self.assertEqual([x[0] for x in insertion_sort([(3, 0), (1, 0), (2, 0)])], [1, 2, 3])

    def test_fit(self):
        model = MyKNN(k=3).fit(self.X, self.y)
        self.assertEqual(model.X_train.shape, (4, 2))

    def test_predict(self):
        model = MyKNN(k=3).fit(self.X, self.y)
        pred = model.predict(np.array([[0., 0.], [5., 5.]]))
        self.assertTrue(np.array_equal(pred, np.array([0, 1])))

    def test_score(self):
        model = MyKNN(k=3).fit(self.X, self.y)
        self.assertEqual(model.score(self.X, self.y), 1.0)

    def test_ai_knn(self):
        model = AIKNN(k=3).fit(self.X, self.y)
        pred = model.predict(np.array([[0., 0.], [5., 5.]]))
        self.assertTrue(np.array_equal(pred, np.array([0, 1])))

    def test_weighted_knn(self):
        model = WeightedKNN(k=3).fit(self.X, self.y)
        pred = model.predict(np.array([[0., 0.], [5., 5.]]))
        self.assertTrue(np.array_equal(pred, np.array([0, 1])))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestKNN)
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)

print("Tests run:", test_result.testsRun)
print("Failures:", len(test_result.failures))
print("Errors:", len(test_result.errors))


In [ ]:
results.to_csv("lab6_knn_results.csv", index=False)
k_results.to_csv("lab6_k_comparison.csv", index=False)
weighted_results.to_csv("lab6_weighted_knn_results.csv", index=False)

print(results)
print(k_results)
print(weighted_results)
